In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated

from langgraph.checkpoint.memory import InMemorySaver

In [3]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.5 
)

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explaination: str

In [6]:
def generate_joke(state: JokeState):
    prompt = f"generate a joke on the topic: {state['topic']}"

    res = llm.invoke(prompt).content

    return {'joke': res}


def generate_explaination(state: JokeState):

    prompt = f"generate an explaination for the  joke: '''{state['joke']}'''. keep the explaination under 50 words "

    res = llm.invoke(prompt).content

    return {'explaination': res}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": '1'}}
initial_state = {
    'topic': "Roronoa zoro"
}
final_state = workflow.invoke(initial_state, config=config1)

In [10]:
final_state['joke']

"Why did Zoro bring a compass to the *Thousand Sunny's galley*?\n\nHe was afraid he'd get lost trying to find his way back to the *deck*!"

In [11]:
workflow.get_state(config=config1)

StateSnapshot(values={'topic': 'Roronoa zoro', 'joke': "Why did Zoro bring a compass to the *Thousand Sunny's galley*?\n\nHe was afraid he'd get lost trying to find his way back to the *deck*!", 'explaination': "Zoro is notoriously terrible with directions in One Piece. The joke plays on his comical inability to navigate, implying he'd get lost even on his own ship, the Thousand Sunny, just trying to find his way from the galley back to the deck."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f190b8e-248d-6394-8002-85214046f16c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-05T10:31:57.000161+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f190b8e-1346-66b7-8001-5d97a1a85335'}}, tasks=(), interrupts=())

In [12]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'Roronoa zoro', 'joke': "Why did Zoro bring a compass to the *Thousand Sunny's galley*?\n\nHe was afraid he'd get lost trying to find his way back to the *deck*!", 'explaination': "Zoro is notoriously terrible with directions in One Piece. The joke plays on his comical inability to navigate, implying he'd get lost even on his own ship, the Thousand Sunny, just trying to find his way from the galley back to the deck."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f190b8e-248d-6394-8002-85214046f16c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-05T10:31:57.000161+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f190b8e-1346-66b7-8001-5d97a1a85335'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Roronoa zoro', 'joke': "Why did Zoro bring a compass to the *Thousand Sunny's galley*?\n\nHe was afraid he'd get lost tryi